### Ansluter till databasen med hjälp av SQLAlchemy

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

server_name = 'localhost'
database_name = 'Bokhandel'

connection_string = (
    f"DRIVER=ODBC Driver 18 for SQL Server;"
    f"SERVER={server_name};"
    f"DATABASE={database_name};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

url_string = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": connection_string}
)

try:

    engine = create_engine(url_string)

    with engine.connect() as connection:
        print(f'Successfully connected to {database_name}!')

except Exception as e:

    print('Error while connecting to database:\n')
    print(e)

Successfully connected to Bokhandel!


### Kontrollera hur många exemplar som finns av varje bok i varje butik

In [38]:
from sqlalchemy import text

search_word = input(" search for a book :")

query = text(""" select

    b.Titel,

    bt.ButiksNamn,

    ISNULL(ls.Antal, 0) AS Antal

from Böcker b

LEFT JOIN LagerSaldo ls
    ON b.ISBN13 = ls.ISBN13

LEFT JOIN Butiker bt
    ON ls.ButikID = bt.butikID
             
 where
    b.Titel like :search

ORDER BY
    b.Titel;
""")

with engine.connect() as conn:

    result = conn.execute(
        query,
        {"search": f"%{search_word}%"}
    )

    rows = result.fetchall()

    for row in rows:
        print(row)

('Harry Potter and the Chamber of Secrets', 'Adlibris', 8)
("Harry Potter and the Philosopher's Stone", 'Adlibris', 10)
("Harry Potter and the Philosopher's Stone", 'Akademibokhandeln', 12)
("Harry Potter and the Philosopher's Stone", 'The English Bookshop', 5)
('Murder on the Orient Express', 'Pocket Shop', 4)
('The Da Vinci Code', 'Bokus', 5)
('The Hobbit', 'Akademibokhandeln', 0)


### Visa resultat som DataFrame

In [39]:
import pandas as pd

df = pd.read_sql_query(
    query,
    con=engine,
    params={"search": f"%{search_word}%"}
)

display(df)

,Titel,ButiksNamn,Antal
0,Harry Potter and the Chamber of Secrets,Adlibris,8
1,Harry Potter and the Philosopher's Stone,Adlibris,10
2,Harry Potter and the Philosopher's Stone,Akademibokhandeln,12
3,Harry Potter and the Philosopher's Stone,The English Bookshop,5
4,Murder on the Orient Express,Pocket Shop,4
5,The Da Vinci Code,Bokus,5
6,The Hobbit,Akademibokhandeln,0
